# Análise exploratória

## Inventário da base

Objetivo: carregar os dados originais, separar a fauna silvestre e verificar o tamanho e a estrutura da base.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
sys.path.append(str(ROOT))

from src.data.data_preparation import load_raw_data, filter_wildlife
from src.analysis.eda_inventory import inventory
from src.analysis.data_quality import quality_summary, duplicate_count
from src.analysis.eda_distributions import count_by_column, monthly_counts, plot_counts
from src.analysis.eda_spatial import coordinate_summary, grid_counts, plot_spatial
from src.data.model_dataset import classification_dataset

In [ ]:
df = load_raw_data()
wildlife = filter_wildlife(df)

print(f'Registros totais: {len(df):,}'.replace(',', '.'))
print(f'Atributos: {df.shape[1]}')
print(f'Registros silvestres: {len(wildlife):,}'.replace(',', '.'))
print(f'Registros domésticos: {len(df) - len(wildlife):,}'.replace(',', '.'))

In [ ]:
inventory(wildlife)

## Interpretação

O conjunto usado nos experimentos deve conter apenas os registros classificados como `silvestre`. Os atributos identificadores e as variáveis que representam informação posterior à ocorrência não serão definidos como preditores automaticamente. Essa decisão será detalhada nas próximas análises.

## Qualidade dos dados

Verificação de valores vazios, registros `nao_identificado` e linhas duplicadas.

In [ ]:
quality_summary(wildlife)

In [ ]:
print(f'Registros duplicados: {duplicate_count(wildlife)}')

## Distribuição dos registros

Aqui verificamos como os registros se distribuem ao longo do tempo e entre os grupos taxonômicos e unidades de conservação.

In [ ]:
count_by_column(wildlife, 'ano')

In [ ]:
monthly_counts(wildlife)

In [ ]:
count_by_column(wildlife, 'classe')

In [ ]:
count_by_column(wildlife, 'uc')

## Distribuição espacial

As coordenadas estão em um sistema plano. Nesta análise verificamos seus limites e agrupamos os registros em células de 500 metros.

In [ ]:
coordinate_summary(wildlife)

In [ ]:
grid_counts(wildlife, cell_size=500).head(20)

## Visualizações

Os gráficos ajudam a identificar concentração temporal, taxonômica e espacial.

In [ ]:
figure, _ = plot_counts(count_by_column(wildlife, 'classe'), 'classe', 'Ocorrências por classe')
figure.savefig(FIGURES_DIR / 'ocorrencias_por_classe.png', dpi=160)
figure

In [ ]:
figure, _ = plot_counts(monthly_counts(wildlife), 'mes', 'Ocorrências por mês')
figure.savefig(FIGURES_DIR / 'ocorrencias_por_mes.png', dpi=160)
figure

In [ ]:
figure, _ = plot_spatial(wildlife)
figure.savefig(FIGURES_DIR / 'distribuicao_espacial.png', dpi=160)
figure

### Gráficos gerados

![Ocorrências por classe](../figures/ocorrencias_por_classe.png)

![Ocorrências por mês](../figures/ocorrencias_por_mes.png)

![Ocorrências por unidade de conservação](../figures/ocorrencias_por_uc.png)

![Distribuição espacial](../figures/distribuicao_espacial.png)

## Gráficos dos modelos

Os gráficos abaixo são gerados pelo pipeline principal e incorporados ao notebook para facilitar a apresentação dos resultados.

### MLP de classificação

![Curvas da MLP de classificação](../reports/generated/mlp_training_curves.png)

### MLP de regressão

![Curvas da MLP de regressão](../reports/generated/mlp_regression_training_curves.png)

![Previsões da MLP de regressão](../reports/generated/mlp_regression_predictions.png)

### Otimização com Optuna

![Histórico do Optuna](../reports/generated/classification_phase5_optuna_history.png)

### Regularização

![MLP sem regularização](../reports/generated/without_regularization_training_curves.png)

![MLP com regularização L2](../reports/generated/with_l2_regularization_training_curves.png)

## Definição do conjunto de classificação

Para a classificação multiclasse, `classe` será a variável alvo. Os identificadores e atributos taxonômicos posteriores à ocorrência não entram como variáveis de entrada.

In [ ]:
X_class, y_class = classification_dataset(wildlife)
print(f'Entradas: {X_class.shape}')
print(f'Alvo: {y_class.shape}')
y_class.value_counts()

## Conclusões da análise exploratória

A base possui registros distribuídos em diferentes anos, meses, classes e posições espaciais. A forte predominância de aves deve ser considerada na avaliação dos modelos, e as coordenadas permitem criar análises por segmentos ou células espaciais. Os atributos de entrada e a variável alvo foram definidos sem usar informações posteriores ao atropelamento.